# Homework 06 (2026) - Spark Answers Notebook

This notebook reproduces the HW06 answers using local Spark.

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import json
import os
import socket

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'Makefile').exists() and (p / 'HOMEWORKS').exists():
            return p
    raise RuntimeError('Could not locate repo root from current working directory')

def pick_col(columns, candidates, label):
    existing = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in existing:
            return existing[c.lower()]
    raise RuntimeError(f'Missing {label} column. Available: {columns}')

def closest_option(value, options):
    return min(options, key=lambda x: abs(x - value))

In [3]:
root = find_repo_root(Path.cwd())
data_dir = root / 'data' / 'raw' / 'hw06'
output_dir = root / 'data' / 'processed' / 'hw06' / 'yellow_2025_11_repartitioned_4'
yellow_path = data_dir / 'yellow_tripdata_2025-11.parquet'
zones_path = data_dir / 'taxi_zone_lookup.csv'

#print(f'ROOT: {root}')
print(f'YELLOW exists: {yellow_path.exists()}')
print(f'ZONES exists: {zones_path.exists()}')

YELLOW exists: True
ZONES exists: True


In [4]:
# Set SPARK_LOCAL_IP explicitly to avoid loopback-hostname warning on some Linux hosts.
if 'SPARK_LOCAL_IP' not in os.environ:
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        sock.connect(('8.8.8.8', 80))
        os.environ['SPARK_LOCAL_IP'] = sock.getsockname()[0]
        sock.close()
    except Exception:
        os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

spark = SparkSession.builder.appName('hw06-notebook').master('local[*]').getOrCreate()
spark_version = spark.version
ui_url = spark.sparkContext.uiWebUrl or ''
ui_port = urlparse(ui_url).port if ui_url else None

print('Spark version:', spark_version)
print('Spark UI URL:', ui_url)
print('SPARK_LOCAL_IP:', os.environ.get('SPARK_LOCAL_IP'))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 20:10:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.5
Spark UI URL: http://192.168.1.128:4040
SPARK_LOCAL_IP: 192.168.1.128


In [ ]:
# Q1 and Q5 - SQL alternatives
# Q1 via Spark SQL query
try:
    q1_sql_version = spark.sql("SELECT version() AS spark_version").first()["spark_version"]
except Exception:
    q1_sql_version = spark.sql(f"SELECT '{spark_version}' AS spark_version").first()["spark_version"]
print('Q1 SQL Spark version:', q1_sql_version)

# Q5 via Spark SQL query (pure SQL literal, no Python-worker path)
q5_sql_port = spark.sql(
    f"""
    SELECT CAST(element_at(split('{ui_url}', ':'), -1) AS INT) AS ui_port
    """
).first()['ui_port']
print('Q5 SQL UI port:', q5_sql_port)

In [5]:
df = spark.read.parquet(str(yellow_path))
pickup_col = pick_col(df.columns, ['trip_pickup_date_time', 'tpep_pickup_datetime'], 'pickup datetime')
dropoff_col = pick_col(df.columns, ['trip_dropoff_date_time', 'tpep_dropoff_datetime'], 'dropoff datetime')
pu_col = pick_col(df.columns, ['PULocationID', 'pulocationid'], 'pickup location id')

zones_df = spark.read.option('header', True).option('inferSchema', True).csv(str(zones_path))
zone_id_col = pick_col(zones_df.columns, ['LocationID', 'locationid'], 'zone location id')
zone_name_col = pick_col(zones_df.columns, ['Zone', 'zone'], 'zone name')

print('Rows in yellow:', df.count())

Rows in yellow: 4181444


In [ ]:
# Prepare SQL temp views used by query-based alternatives.
df.createOrReplaceTempView('yellow_trips')
zones_df.select(
    F.col(zone_id_col).alias('location_id'),
    F.col(zone_name_col).alias('zone_name')
).createOrReplaceTempView('taxi_zones')
print('SQL views ready: yellow_trips, taxi_zones')

In [6]:
if output_dir.exists():
    for p in output_dir.rglob('*'):
        if p.is_file():
            p.unlink()
    for p in sorted(output_dir.rglob('*'), reverse=True):
        if p.is_dir():
            p.rmdir()
output_dir.mkdir(parents=True, exist_ok=True)

df.repartition(4).write.mode('overwrite').parquet(str(output_dir))
parquet_files = sorted(output_dir.rglob('*.parquet'))
sizes_mb = [p.stat().st_size / (1024 * 1024) for p in parquet_files]
q2_avg_mb = sum(sizes_mb) / len(sizes_mb)
q2_option = closest_option(q2_avg_mb, [6.0, 25.0, 75.0, 100.0])

print('Q2 average MB:', round(q2_avg_mb, 2))
print('Q2 closest option:', q2_option)

Q2 average MB: 24.42
Q2 closest option: 25.0


In [ ]:
# Q2 - SQL alternative (average parquet part size from written output)
q2_sql_query = f"""
SELECT AVG(size_bytes) / (1024 * 1024) AS avg_file_size_mb
FROM (
    SELECT length(content) AS size_bytes
    FROM binaryFile.`{output_dir.as_posix()}/*.parquet`
)
"""
q2_avg_mb_sql = spark.sql(q2_sql_query).first()['avg_file_size_mb']
q2_option_sql = closest_option(float(q2_avg_mb_sql), [6.0, 25.0, 75.0, 100.0])
print('Q2 SQL average MB:', round(float(q2_avg_mb_sql), 2))
print('Q2 SQL closest option:', q2_option_sql)

In [7]:
q3_count = df.where(F.to_date(F.col(pickup_col)) == F.lit('2025-11-15')).count()
print('Q3 trips on 2025-11-15:', q3_count)

Q3 trips on 2025-11-15: 162604


In [ ]:
# Q3 - SQL alternative
q3_sql_query = f"""
SELECT COUNT(*) AS trips_started_2025_11_15
FROM yellow_trips
WHERE TO_DATE(`{pickup_col}`) = DATE '2025-11-15'
"""
q3_sql_count = spark.sql(q3_sql_query).first()['trips_started_2025_11_15']
print('Q3 SQL trips on 2025-11-15:', q3_sql_count)

In [8]:
trip_hours_df = df.withColumn(
    'trip_hours',
    (F.unix_timestamp(F.col(dropoff_col)) - F.unix_timestamp(F.col(pickup_col))) / F.lit(3600.0),
)
q4_hours = trip_hours_df.agg(F.max('trip_hours').alias('max_trip_hours')).collect()[0]['max_trip_hours']
q4_hours = float(q4_hours) if q4_hours is not None else float('nan')
print('Q4 longest trip hours:', round(q4_hours, 1))

Q4 longest trip hours: 90.6


In [ ]:
# Q4 - SQL alternative
q4_sql_query = f"""
SELECT ROUND(MAX((unix_timestamp(`{dropoff_col}`) - unix_timestamp(`{pickup_col}`)) / 3600.0), 1) AS longest_trip_hours
FROM yellow_trips
"""
q4_sql_hours = spark.sql(q4_sql_query).first()['longest_trip_hours']
print('Q4 SQL longest trip hours:', q4_sql_hours)

In [9]:
q6_row = (
    df.groupBy(F.col(pu_col).alias('location_id')).count()
    .join(
        zones_df.select(F.col(zone_id_col).alias('location_id'), F.col(zone_name_col).alias('zone_name')),
        on='location_id',
        how='left'
    )
    .groupBy('zone_name')
    .agg(F.sum('count').alias('trip_count'))
    .orderBy(F.col('trip_count').asc(), F.col('zone_name').asc())
    .first()
)
q6_zone = q6_row['zone_name'] if q6_row else None
print('Q6 least frequent pickup zone:', q6_zone)

Q6 least frequent pickup zone: Arden Heights


In [ ]:
# Q6 - SQL alternative
q6_sql_query = f"""
WITH trip_counts AS (
    SELECT CAST(`{pu_col}` AS INT) AS location_id, COUNT(*) AS trip_count
    FROM yellow_trips
    GROUP BY CAST(`{pu_col}` AS INT)
)
SELECT z.zone_name
FROM trip_counts t
LEFT JOIN taxi_zones z
    ON t.location_id = z.location_id
GROUP BY z.zone_name
ORDER BY SUM(t.trip_count) ASC, z.zone_name ASC
LIMIT 1
"""
q6_sql_zone = spark.sql(q6_sql_query).first()['zone_name']
print('Q6 SQL least frequent pickup zone:', q6_sql_zone)

In [10]:
answers = {
    'q1': spark_version,
    'q2': {'avg_file_size_mb': round(q2_avg_mb, 2), 'closest_option_mb': q2_option},
    'q3': int(q3_count),
    'q4': round(q4_hours, 1),
    'q5': ui_port,
    'q6': q6_zone,
}
print(json.dumps(answers, indent=2))

{
  "q1": "3.5.5",
  "q2": {
    "avg_file_size_mb": 24.42,
    "closest_option_mb": 25.0
  },
  "q3": 162604,
  "q4": 90.6,
  "q5": 4040,
  "q6": "Arden Heights"
}


In [11]:
# Optional cleanup
# spark.stop()